### Import libraries and load data

In [14]:

import pandas as pd
import re
from collections import Counter

# Load input files
skills_df = pd.read_csv("..//data//skill-list.csv")
jobs_df = pd.read_csv("..//data//job-descriptions-tagged.csv")

SKILLS = skills_df["skill"].dropna().str.strip().tolist()

# Check input data
print("Approved skills:", len(SKILLS))
print("Job descriptions:", len(jobs_df))
print(jobs_df.head(3))


Approved skills: 54
Job descriptions: 50
   jd_id                      title          company  \
0  JD001        Data Analyst Intern    Kite Learning   
1  JD002  Frontend Developer Intern  Orbit Logistics   
2  JD003   Backend Developer Intern    Vayu Mobility   

                                         description  \
0  Join Kite Learning as a Data Analyst Intern fo...   
1  Join Orbit Logistics as a Frontend Developer I...   
2  Join Vayu Mobility as a Backend Developer Inte...   

                                        model_tags  
0    Statistics; Microsoft Excel; Teamwork; Python  
1                      TypeScript; JavaScript; CSS  
2  REST APIs; PostgreSQL; Node; Leadership; Docker  


### Define synonym mapping

In [15]:

ALIASES = {
    "Node.js": ["Node", "Node.js"],
    "PostgreSQL": ["Postgres", "PostgreSQL"],
    "Microsoft Excel": ["Excel", "Microsoft Excel"],
    "JavaScript": ["JS", "JavaScript"],
    "REST APIs": ["API", "APIs", "REST API", "REST APIs"],
    "Machine Learning": ["ML", "Machine Learning"],
    "Social Media Marketing": [
        "Social Media", "Social Media Marketing"
    ],
    "Financial Modelling": [
        "Financial Modeling", "Financial Modelling"
    ]
}

OPTIONAL_MARKERS = [
    "bonus if",
    "nice to have",
    "is a plus",
    "plus, not a must",
    "not a must"
]

def normalize_text(text):
    """Standardise whitespace and case."""
    return re.sub(r"\s+", " ", str(text).strip()).lower()


def get_skill_pattern(skill):
    """Build a regex pattern for a canonical skill and its aliases."""

    terms = ALIASES.get(skill, [skill])

    # JS must not match the .js suffix in Node.js.
    alternatives = []
    for term in sorted(terms, key=len, reverse=True):
        escaped = re.escape(term)

        if term == "JS":
            alternatives.append(
                r"(?<![a-z0-9.+])" + escaped + r"(?![a-z0-9+#])"
            )
        elif term in ["API", "APIs", "REST API", "REST APIs"]:
            continue
        else:
            alternatives.append(
                r"(?<![a-z0-9+#])" + escaped + r"(?![a-z0-9+#])"
            )

    if skill == "REST APIs":
        alternatives.append(
            r"(?<![a-z0-9+#])apis?(?![a-z0-9+#])"
        )

    return re.compile(
        "|".join(alternatives),
        flags=re.IGNORECASE
    )


SKILL_PATTERNS = {
    skill: get_skill_pattern(skill)
    for skill in SKILLS
}


def normalize_tag(tag):
    """Convert a predicted tag to its approved skill name."""

    tag = normalize_text(tag)

    for skill, pattern in SKILL_PATTERNS.items():
        if pattern.fullmatch(tag):
            return skill

    return None


### Extract required and optional skills

In [16]:

def extract_skills(description):

    description = re.sub(
        r"\s+", " ", str(description)
    )

    # Split into sentence-like sections
    chunks = re.split(
        r"(?<=[.!?])\s+", description
    )

    skill_status = {}

    for skill, pattern in SKILL_PATTERNS.items():

        statuses = []

        for chunk in chunks:

            matches = list(pattern.finditer(chunk))

            if not matches:
                continue

            # Avoid treating User Research as a separate
            # occurrence of Research.
            if skill == "Research":

                user_research_matches = list(
                    SKILL_PATTERNS["User Research"].finditer(chunk)
                )

                if all(
                    any(
                        u.start() <= m.start()
                        and u.end() >= m.end()
                        for u in user_research_matches
                    )
                    for m in matches
                ):
                    continue

            lower_chunk = chunk.lower()

            optional = any(
                marker in lower_chunk
                for marker in OPTIONAL_MARKERS
            )

            statuses.append(
                "optional" if optional else "required"
            )

        if statuses:
            # A required mention takes priority if the
            # same skill appears in both contexts.
            skill_status[skill] = (
                "required"
                if "required" in statuses
                else "optional"
            )

    return skill_status


### Evaluate model predictions

In [17]:

results = []

for _, row in jobs_df.iterrows():

    jd_id = row["jd_id"]
    description = row["description"]
    model_tags = str(row["model_tags"])

    # Extract reference skills
    expected = extract_skills(description)

    # Split and normalise predicted tags
    raw_tags = [
        tag.strip()
        for tag in model_tags.split(";")
        if tag.strip()
    ]

    predicted = set()
    invalid_tags = []

    for tag in raw_tags:

        canonical = normalize_tag(tag)

        if canonical:
            predicted.add(canonical)
        else:
            invalid_tags.append(tag)

    expected_set = set(expected.keys())

    required_set = {
        skill for skill, status in expected.items()
        if status == "required"
    }

    optional_set = {
        skill for skill, status in expected.items()
        if status == "optional"
    }

    # Classification
    correct = predicted & expected_set

    wrong = (
        predicted - expected_set
    ) | set(invalid_tags)

    missing = required_set - predicted

    # Store row-level results
    results.append({

        "jd_id": jd_id,

        "required_skills": "; ".join(sorted(required_set)),

        "optional_skills": "; ".join(sorted(optional_set)),

        "predicted_tags": "; ".join(sorted(predicted)),

        "correct_tags": "; ".join(sorted(correct)),

        "wrong_tags": "; ".join(sorted(wrong)),

        "missing_required": "; ".join(sorted(missing)),

        "correct_count": len(correct),

        "wrong_count": len(wrong),

        "missing_count": len(missing),

        "required_count": len(required_set),

        "predicted_count": len(raw_tags)

    })


evaluation_df = pd.DataFrame(results)

# Display complete evaluation
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

print(evaluation_df.to_string(index=False))


jd_id                                             required_skills        optional_skills                                                                                  predicted_tags                                                                  correct_tags             wrong_tags                  missing_required  correct_count  wrong_count  missing_count  required_count  predicted_count
JD001                         Microsoft Excel; Python; Statistics          Data Cleaning                                                             Microsoft Excel; Python; Statistics                                           Microsoft Excel; Python; Statistics               Teamwork                                                3            1              0               3                4
JD002                          CSS; JavaScript; React; TypeScript                   HTML                                                                     CSS; JavaScript; TypeScript                        

### Calculate evaluation metrics

In [18]:

total_predicted = evaluation_df["predicted_count"].sum()

correct_tags = evaluation_df["correct_count"].sum()

wrong_tags = evaluation_df["wrong_count"].sum()

total_required = evaluation_df["required_count"].sum()

missing_skills = evaluation_df["missing_count"].sum()

# Correct required skills, excluding correctly tagged
# optional skills from the recall calculation.
required_correct = 0

for _, row in jobs_df.iterrows():

    expected = extract_skills(row["description"])

    required = {
        skill for skill, status in expected.items()
        if status == "required"
    }

    predicted = {
        normalize_tag(tag)
        for tag in str(row["model_tags"]).split(";")
        if tag.strip()
    }

    predicted.discard(None)

    required_correct += len(required & predicted)


precision = (
    correct_tags / total_predicted
    if total_predicted else 0
)

recall = (
    required_correct / total_required
    if total_required else 0
)

missing_rate = (
    missing_skills / total_required
    if total_required else 0
)

print("\n========== EVALUATION SUMMARY ==========")

print("Job descriptions:", len(evaluation_df))

print("Total predicted tags:", total_predicted)

print("Correct tags:", correct_tags)

print("Wrong tags:", wrong_tags)

print("Total required skills:", total_required)

print("Correct required skills:", required_correct)

print("Missing required skills:", missing_skills)

print("\n========== METRICS ==========")

print(f"Tag Precision: {precision:.2%}")

print(f"Required Skill Recall: {recall:.2%}")

print(f"Missing Skill Rate: {missing_rate:.2%}")



========== EVALUATION SUMMARY ==========
Job descriptions: 50
Total predicted tags: 200
Correct tags: 171
Wrong tags: 29
Total required skills: 181
Correct required skills: 153
Missing required skills: 28

========== METRICS ==========
Tag Precision: 85.50%
Required Skill Recall: 84.53%
Missing Skill Rate: 15.47%


### Export the evaluation sheet

In [20]:

# Export row-level labels
evaluation_df.to_csv("..//results//skill_tag_evaluation.csv", index=False)

# Review descriptions with errors
error_rows = evaluation_df[
    (evaluation_df["wrong_count"] > 0) |
    (evaluation_df["missing_count"] > 0)
]

print(error_rows[
    ["jd_id", "correct_tags", "wrong_tags", "missing_required"]
].to_string(index=False))


jd_id                                                                  correct_tags             wrong_tags                  missing_required
JD001                                           Microsoft Excel; Python; Statistics               Teamwork                                  
JD002                                                   CSS; JavaScript; TypeScript                                                    React
JD003                                                   Docker; Node.js; PostgreSQL  Leadership; REST APIs                    Git; REST APIs
JD004                                     Content Writing; Research; SEO; WordPress                    SAP                                  
JD005 Canva; Copywriting; Email Marketing; Google Analytics; Social Media Marketing       Machine Learning                                  
JD007                         Accounting; Financial Modelling; GST; Microsoft Excel             Kubernetes                                  
JD008        